# 1. Performance Analysis

**Objective**: Evaluate the trained model on the Test Set using standard metrics and detailed error analysis.

**Metrics**:
- **ROC-AUC**: General discrimination ability.
- **Precision/Recall**: Critical for this business case (avoiding False Positives = investing in losing projects).
- **Calibration**: Are the probabilities trustworthy?


In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from app.src import config

SAVE_DIR = config.RESULTS_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

# Load Data and Predicted Probabilities (from Training Step)
# Since train_model.py runs on Test Set, we can reload its results or re-predict.
# For analysis depth, let's load model and re-predict validation set.
test_df = pd.read_csv(config.TEST_DATA_PATH)
y_test = test_df[config.TARGET_COL]
X_test = test_df.drop(config.TARGET_COL, axis=1)

model = joblib.load(config.MODEL_PATH)
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
# ROC Curve Analysis
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.savefig(os.path.join(SAVE_DIR, '../results/roc_curve_analysis.png'))
plt.show()

## 1.1 Error Analysis (Qualitative)
We investigate the "Confused" cases:
- **False Positives (FP)**: Model said SUCCESS (Prob > 0.6), but failed. (Worst case for investor).
- **False Negatives (FN)**: Model said FAIL, but it succeeded. (Missed opportunity).

In [ ]:
# Create Analysis DataFrame
analysis_df = X_test.copy()
analysis_df['actual'] = y_test
analysis_df['prob'] = y_prob
analysis_df['pred'] = (y_prob >= config.DECISION_THRESHOLD).astype(int)

# Extract False Positives
# Note: We need original text features to understand WHY. 
# If we dropped them, we might need to join back with Raw Data using Index if preserved.
# Assuming indices match:
raw_test_data = pd.read_csv(config.RAW_DATA_PATH).iloc[X_test.index]
analysis_df = analysis_df.join(raw_test_data[['name', 'blurb', 'category']], rsuffix='_raw')

false_positives = analysis_df[(analysis_df['pred'] == 1) & (analysis_df['actual'] == 0)].sort_values(by='prob', ascending=False)

print("Top 5 False Positives (High Confidence Failures):")
for i, row in false_positives.head(5).iterrows():
    print(f"- Prob: {row['prob']:.2f} | Name: {row['name']} | Cat: {row['category']}")
    print(f"  Blurb: {row['blurb']}\n")

**Insight from Errors**:
- Often, False Positives are high-quality hardware projects that look professional (good text, high goal) but failed due to external factors (marketing, pricing) not captured in the dataset.
- False Negatives might be "viral" jokes or niche community projects that look like "garbage" to the model but succeeded.

# 2. Business Impact (ROI)
Translating model performance into money. If we invest $100 in every predicted success, what is the return?

In [ ]:
# Simple ROI Simulation
thresholds = [0.5, 0.6, 0.7, 0.8]
results = []

for t in thresholds:
    preds = (y_prob >= t)
    invested_count = preds.sum()
    success_count = (preds & (y_test == 1)).sum()
    
    # Assoc. Profit (Hypothetical: Fail = -$100, Success = +$20)
    profit = (success_count * 20) - ((invested_count - success_count) * 100)
    roi = (profit / (invested_count * 100)) * 100 if invested_count > 0 else 0
    
    results.append({'Threshold': t, 'Invested': invested_count, 'Profit': profit, 'ROI %': roi})

roi_df = pd.DataFrame(results)
print(roi_df)

plt.figure(figsize=(8, 5))
sns.lineplot(data=roi_df, x='Threshold', y='ROI %', marker='o')
plt.title('ROI vs Decision Threshold')
plt.axhline(0, color='red', linestyle='--')
plt.savefig(os.path.join(SAVE_DIR, '../results/roi_analysis.png'))
plt.show()